<a href="https://colab.research.google.com/github/armandoordonez/AI-Engineering/blob/main/rag_chromadb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import ipykernel
import requests
from notebook.notebookapp import list_running_servers
from urllib.parse import urljoin

def limpiar_este_notebook_actual():
    try:
        # Obtener el ID del kernel actual en ejecución
        kernel_id = ipykernel.connect.get_connection_file().split("kernel-")[-1].split(".json")[0]

        # Buscar los servidores locales activos de Jupyter/Colab para mapear el archivo actual
        for server in list_running_servers():
            try:
                sesiones_url = urljoin(server['url'], 'api/sessions')
                respuesta = requests.get(sesiones_url, params={'token': server.get('token', '')})
                if respuesta.status_code == 200:
                    for sesion in respuesta.json():
                        if sesion['kernel']['id'] == kernel_id:
                            ruta_notebook_actual = sesion['notebook']['path']

                            # Leer y limpiar los metadatos de este archivo .ipynb en particular
                            with open(ruta_notebook_actual, 'r', encoding='utf-8') as f:
                                data = json.load(f)

                            if 'metadata' in data and 'widgets' in data['metadata']:
                                del data['metadata']['widgets']
                                with open(ruta_notebook_actual, 'w', encoding='utf-8') as f:
                                    json.dump(data, f, indent=1, ensure_ascii=False)
                                print(f"-> ¡Clave 'metadata.widgets' eliminada con éxito de este notebook ({ruta_notebook_actual})!")
                            else:
                                print("-> El notebook actual ya se encuentra sin de metadatos conflictivos de widgets.")
                            return
            except Exception:
                continue
    except Exception as e:
        # Método alternativo genérico en caso de que las APIs de sesión estén restringidas en el sandbox de Colab
        import os
        archivos = [f for f in os.listdir('.') if f.endswith('.ipynb')]
        for arch in archivos:
            with open(arch, 'r', encoding='utf-8') as f:
                data = json.load(f)
            if 'metadata' in data and 'widgets' in data['metadata']:
                del data['metadata']['widgets']
                with open(arch, 'w', encoding='utf-8') as f:
                    json.dump(data, f, indent=1, ensure_ascii=False)
                print(f"-> Se limpió el archivo local: {arch}")
                return
        print("Guarda el archivo del notebook (.ipynb) en el menú Archivo > Guardar, y vuelve a ejecutar esta celda.")

limpiar_este_notebook_actual()

In [ ]:
# Instalar las librerías necesarias para el sistema RAG:
# - chromadb: Base de datos vectorial para almacenar y buscar embeddings.
# - sentence-transformers: Para generar representaciones vectoriales (embeddings) del texto.
# - transformers: Para cargar y utilizar el modelo de lenguaje generativo.
# - torch: El framework de aprendizaje profundo que respalda a los modelos.
!pip install chromadb sentence-transformers transformers torch

In [ ]:
# Definición de la base de conocimientos inicial
# Estos documentos contienen la información que el sistema consultará para responder preguntas
documents = [
    "La inteligencia artificial es una rama de la informática.",
    "Los modelos de lenguaje generativo son muy populares en IA.",
    "ChromaDB es una base de datos vectorial eficiente.",
    "Python es un lenguaje de programación ampliamente utilizado."
]

In [ ]:
from sentence_transformers import SentenceTransformer

# 1. Cargar un modelo pre-entrenado de SentenceTransformers
# 'all-MiniLM-L6-v2' es un modelo ligero y eficiente para crear embeddings de texto
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Convertir los documentos de texto en vectores numéricos (embeddings)
# Cada documento se transforma en un vector que captura su significado semántico
embeddings = model.encode(documents)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
import chromadb

# 1. Inicializar el cliente de ChromaDB con persistencia local
# Los datos se guardarán en la carpeta 'chroma_db' para que no se pierdan al reiniciar
client = chromadb.PersistentClient(path="chroma_db")

# 2. Obtener o crear la colección (evita el error si ya existe en el almacenamiento persistente)
collection = client.get_or_create_collection(name="knowledge_base")

# 3. Almacenar los embeddings y los textos correspondientes en la colección
for i, (doc, emb) in enumerate(zip(documents, embeddings)):
    collection.add(
        ids=[str(i)],              # ID único en formato string
        embeddings=[emb.tolist()],  # El embedding debe convertirse de array NumPy a lista estándar de Python
        documents=[doc]            # El texto original asociado al embedding
    )

In [ ]:
def retrieve(query, top_k=2):
    # 1. Convertir la consulta del usuario en un embedding usando el mismo modelo
    query_embedding = model.encode(query)

    # 2. Buscar en ChromaDB los 'top_k' documentos más cercanos vectorialmente a la consulta
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    # 3. Retornar la lista de textos recuperados
    return results['documents'][0]

In [ ]:
from transformers import pipeline

# 1. Cargar el modelo de generación de texto
generator = pipeline("text-generation", model="EleutherAI/gpt-neo-125M")

def generate_response(query, retrieved_docs):
    # 2. Unir los documentos recuperados en un solo bloque de texto que servirá de contexto
    context = " ".join(retrieved_docs)

    # 3. Construir una plantilla de Prompt muy estricta
    prompt = (
        f"Instrucciones: Responde a la pregunta basándote únicamente en el Contexto provisto. "
        f"Si la respuesta no se menciona de manera directa en el Contexto, responde estrictamente: 'No poseo esa información en mi base de conocimientos'.\n"
        f"Contexto: {context}\n"
        f"Pregunta: {query}\n"
        f"Respuesta:"
    )

    # 4. Generar la respuesta de forma ultra-determinista
    response = generator(
        prompt,
        max_new_tokens=50,
        num_return_sequences=1,
        truncation=True,
        do_sample=False
    )

    # 5. Extraer y limpiar el resultado para devolver solo la respuesta generada
    generated_text = response[0]['generated_text']

    # Eliminamos el prompt original de la respuesta generada para quedarnos solo con la respuesta pura
    if "Respuesta:" in generated_text:
        clean_response = generated_text.split("Respuesta:")[-1].strip()
        # Recortamos en el siguiente salto de línea o etiqueta de Contexto para evitar bucles repetitivos
        clean_response = clean_response.split("\n")[0].split("Contexto:")[0].strip()

        # Si el modelo copió múltiples enunciados del contexto que no corresponden a la pregunta,
        # nos quedamos solo con la primera oración relevante o el enunciado correspondiente a la consulta.
        if "." in clean_response:
            sentences = [s.strip() for s in clean_response.split(".") if s.strip()]
            # Filtrar oraciones que no contengan palabras clave de la pregunta (por ejemplo, 'Python')
            keyword = query.replace("¿", "").replace("?", "").split()[-1].lower() # Extrae una palabra clave relevante
            for sentence in sentences:
                if keyword in sentence.lower():
                    return sentence + "."
            return sentences[0] + "."

        return clean_response

    return generated_text

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Definir la función integradora de RAG de nuevo para asegurar que toma la última versión de generate_response
def rag_system(query):
    retrieved_docs = retrieve(query)
    response = generate_response(query, retrieved_docs)
    return response

# Volvemos a probar el sistema RAG para verificar que responde únicamente lo solicitado
query = "¿Qué es Python?"
response = rag_system(query)
print("Pregunta:", query)
print("Respuesta Limpia:", response)

Pregunta: ¿Qué es Python?
Respuesta Limpia: Python es un lenguaje de programación ampliamente utilizado.
